In [ ]:
import subprocess, os, sys, json

# Install system dependencies for Chromium
subprocess.run(['apt-get', 'install', '-y',
    'libatk1.0-0', 'libatk-bridge2.0-0', 'libcups2',
    'libxcomposite1', 'libxdamage1', 'libxfixes3',
    'libxrandr2', 'libgbm1', 'libxkbcommon0',
    'libpango-1.0-0', 'libcairo2', 'libasound2'
], capture_output=True)

# Install Python packages
subprocess.run([sys.executable, '-m', 'pip', 'install',
    'playwright', 'opencv-python-headless', 'pillow', '--quiet'])

# Install Playwright browsers
subprocess.run(['playwright', 'install', 'chromium'])

print('Setup complete!')


In [ ]:
if not os.path.exists('/kaggle/working/amer'):
    subprocess.run(['git', 'clone',
        'https://github.com/amerameryou1-blip/amer.git',
        '/kaggle/working/amer'])
else:
    # Pull latest changes
    subprocess.run(['git', '-C', '/kaggle/working/amer', 'pull'])

candidate_paths = [
    '/kaggle/working/amer/New folder (2)/territorial_bot',
    '/kaggle/working/amer/territorial_bot'
]
BOT_PATH = next((path for path in candidate_paths if os.path.exists(path)), candidate_paths[0])
sys.path.insert(0, BOT_PATH)
print('Repo ready. Bot path:', BOT_PATH)


In [ ]:
config_path = os.path.join(BOT_PATH, 'config.json')
with open(config_path) as f:
    config = json.load(f)

# Ensure all required sections exist
config.setdefault('vision', {
    'map_region': [0, 0, 1366, 768],
    'territory_sample_grid': 10,
    'color_tolerance': 20,
    'min_territory_pixels': 50,
    'use_grayscale_fallback': False,
    'process_width': 384,
    'process_height': 216,
    'playable_area': [310, 80, 1070, 640],
    'troop_bar_sample_pixel': [760, 748],
    'defeat_check_region': [490, 595, 570, 150]
})

config.setdefault('debug', {
    'save_screenshots': False,
    'screenshot_every_n_steps': 100,
    'verbose_logging': False
})

# Kaggle overrides
config['game']['headless'] = True
config['game']['window_width'] = 1366
config['game']['window_height'] = 768
config['training']['checkpoint_dir'] = '/kaggle/working/checkpoints/'
config['training']['log_dir'] = '/kaggle/working/logs/'
config['training']['num_workers'] = 1

os.makedirs('/kaggle/working/checkpoints/', exist_ok=True)
os.makedirs('/kaggle/working/logs/', exist_ok=True)

with open(config_path, 'w') as f:
    json.dump(config, f, indent=2)

# Patch config_loader to skip strict validation
loader_path = os.path.join(BOT_PATH, 'config_loader.py')
with open(loader_path) as f:
    content = f.read()
if '_validate_schema(config, REQUIRED_CONFIG_SCHEMA)' in content:
    content = content.replace(
        '_validate_schema(config, REQUIRED_CONFIG_SCHEMA)',
        'pass  # validation skipped for Kaggle compatibility'
    )
    with open(loader_path, 'w') as f:
        f.write(content)

print('Config ready:', list(config.keys()))


In [ ]:
import subprocess, sys

candidate_main_paths = [
    '/kaggle/working/amer/New folder (2)/territorial_bot/main.py',
    '/kaggle/working/amer/territorial_bot/main.py'
]
main_path = next((path for path in candidate_main_paths if os.path.exists(path)), candidate_main_paths[0])

proc = subprocess.Popen(
    ['python', '-u', main_path, '--mode', 'train', '--workers', '1'],
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True,
    bufsize=1
)

try:
    for line in proc.stdout:
        print(line, end='', flush=True)
except KeyboardInterrupt:
    proc.terminate()
    print('\nTraining stopped by user')


In [ ]:
from IPython.display import FileLink
import os

ckpt = '/kaggle/working/checkpoints/q_table.pkl'
if os.path.exists(ckpt):
    print(f'Checkpoint size: {os.path.getsize(ckpt)/1024:.1f} KB')
    display(FileLink(ckpt))
else:
    print('No checkpoint found yet')
